# Association Rule Learning: Theory, Math, and Implementation

Association Rule Learning is an **unsupervised** machine learning method used to discover interesting relations, hidden patterns, and frequent itemsets within large databases. Unlike clustering which groups rows (data points) together, association rules group **columns (items)** together based on how frequently they co-occur.

---
### 1. Apriori and Eclat Algorithms

These are the most famous algorithms for finding frequent itemsets in a dataset. 
* **Apriori** uses a "Breadth-First Search" approach. It generates candidate itemsets step-by-step (1-item sets, then 2-item sets, etc.) and eliminates candidates that are infrequent. It relies on the **Apriori Property**: *If an itemset is infrequent, all of its supersets must also be infrequent.*
* **Eclat (Equivalence Class Clustering and bottom-up Lattice Traversal)** uses a "Depth-First Search" approach. Instead of looking at transactions to count items, it looks at items and keeps a list of the exact Transaction IDs (TID) where they appear. It calculates frequencies by mathematically intersecting these TID sets.

**Mathematical Foundation (The 3 Metrics):**
Association rules are written in the format $X \rightarrow Y$ (If $X$ occurs, then $Y$ also occurs). To determine if a rule is statistically significant, we use three mathematical metrics:

**1. Support:** How popular is the itemset? It is the probability that a transaction contains both $X$ and $Y$.
$$Support(X \rightarrow Y) = \frac{\text{Transactions containing both } X \text{ and } Y}{\text{Total number of transactions}}$$

**2. Confidence:** How likely is $Y$ to be purchased *given* that $X$ was purchased? It is the conditional probability $P(Y \mid X)$.
$$Confidence(X \rightarrow Y) = \frac{Support(X \rightarrow Y)}{Support(X)}$$

**3. Lift:** How much does the occurrence of $X$ actually increase the probability of $Y$? A Lift of $1$ means $X$ and $Y$ are completely independent. A Lift $> 1$ means they are positively correlated.
$$Lift(X \rightarrow Y) = \frac{Confidence(X \rightarrow Y)}{Support(Y)} = \frac{Support(X \rightarrow Y)}{Support(X) \times Support(Y)}$$

**Example Problem:**
* **Retail / E-commerce:** Market Basket Analysis. Discovering that "People who buy Bread and Eggs are 80% likely to also buy Butter." Stores use this to optimize shelf layout, and websites use it for "Frequently Bought Together" recommendations.

The interactive visualization below simulates a grocery store database. Use the sliders to filter out weak rules. The resulting network graph shows exactly how items are connected!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from ipywidgets import interact, FloatSlider
import itertools

plt.style.use('seaborn-v0_8-darkgrid')

np.random.seed(42)
items = ['Bread', 'Butter', 'Milk', 'Eggs', 'Diapers', 'Beer', 'Cola', 'Chips']
transactions = []

for _ in range(1000):
    basket = []
    if np.random.rand() > 0.3: 
        basket.append('Bread')
        if np.random.rand() > 0.2: basket.append('Butter') # Bread -> Butter
    if np.random.rand() > 0.5: basket.append('Milk')
    if np.random.rand() > 0.6: basket.append('Eggs')
    if np.random.rand() > 0.8: 
        basket.append('Diapers')
        if np.random.rand() > 0.1: basket.append('Beer') # Diapers -> Beer (Classic urban legend)
    if np.random.rand() > 0.5:
        basket.append('Chips')
        if np.random.rand() > 0.4: basket.append('Cola') # Chips -> Cola
        
    transactions.append(basket)

N = len(transactions)
item_counts = {item: 0 for item in items}
for basket in transactions:
    for item in basket:
        item_counts[item] += 1

@interact(
    min_support=FloatSlider(min=0.01, max=0.40, step=0.01, value=0.10, description='Min Support'),
    min_confidence=FloatSlider(min=0.1, max=1.0, step=0.05, value=0.50, description='Min Confidence'),
    min_lift=FloatSlider(min=1.0, max=3.0, step=0.1, value=1.0, description='Min Lift')
)
def plot_market_basket_network(min_support, min_confidence, min_lift):
    rules = []
    
    for A, B in itertools.permutations(items, 2):
        count_A = sum(1 for basket in transactions if A in basket)
        count_B = sum(1 for basket in transactions if B in basket)
        count_AB = sum(1 for basket in transactions if A in basket and B in basket)
        
        support_A = count_A / N
        support_B = count_B / N
        support_AB = count_AB / N
        
        if support_A == 0 or support_B == 0: continue
            
        confidence = support_AB / support_A
        lift = confidence / support_B
        
        if support_AB >= min_support and confidence >= min_confidence and lift >= min_lift:
            rules.append((A, B, support_AB, confidence, lift))
            
    plt.figure(figsize=(10, 7))
    G = nx.DiGraph()
    
    if not rules:
        plt.title(f"No rules found matching criteria.", fontsize=15, color='red')
        plt.axis('off')
        plt.show()
        return

    for rule in rules:
        A, B, supp, conf, lift = rule
        G.add_edge(A, B, weight=lift, conf=conf)

    pos = nx.spring_layout(G, seed=42, k=1.5)
    
    nx.draw_networkx_nodes(G, pos, node_size=2500, node_color='skyblue', edgecolors='black')
    nx.draw_networkx_labels(G, pos, font_size=11, font_weight='bold')
    
    edges = G.edges()
    weights = [G[u][v]['weight'] * 2 for u, v in edges] # Lift dictates color
    confidences = [G[u][v]['conf'] * 4 for u, v in edges] # Confidence dictates thickness
    
    edge_draw = nx.draw_networkx_edges(G, pos, edgelist=edges, width=confidences, 
                                       edge_color=weights, edge_cmap=plt.cm.Reds, 
                                       arrowsize=20, arrowstyle='->', connectionstyle="arc3,rad=0.1")
    
    sm = plt.cm.ScalarMappable(cmap=plt.cm.Reds, norm=plt.Normalize(vmin=min_lift, vmax=max(weights)/2))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=plt.gca(), fraction=0.046, pad=0.04)
    cbar.set_label('Lift (Rule Strength)', rotation=270, labelpad=15, fontweight='bold')
    
    plt.title("Market Basket Analysis: Association Rules Network", fontsize=15, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"Discovered {len(rules)} valid rules:")
    for rule in sorted(rules, key=lambda x: x[4], reverse=True): # Sort by Lift
        print(f"Rule: {rule[0]:<7} -> {rule[1]:<7} | Support: {rule[2]:.2f} | Confidence: {rule[3]:.2f} | Lift: {rule[4]:.2f}")

interactive(children=(FloatSlider(value=0.1, description='Min Support', max=0.4, min=0.01, step=0.01), FloatSl…

Market Basket Analysis can generate thousands of rules in a real-world scenario. Instead of reading through massive tables of data, a **Directed Network Graph** is one of the most powerful ways to visualize these hidden relationships.

Here is how to interpret the visual elements of the graph:

**1. The Nodes (Blue Bubbles)**
* Each bubble represents a single **Item** (or product) in the dataset. 
* If an item does not meet the minimum support threshold (meaning it is rarely bought), it will completely disappear from the graph.

**2. The Arrows (Directed Edges)**
* The arrows represent the **Association Rule** direction ($X \rightarrow Y$).
* For example, an arrow pointing from *Diapers* to *Beer* means "If a customer buys Diapers, they are likely to buy Beer." 
* *Note: The reverse is not always true! An arrow from A to B does not guarantee an arrow from B to A.*

**3. Arrow Thickness (Confidence)**
* The **thickness** of the line represents the rule's **Confidence**.
* A very thick, heavy line means the rule is highly reliable. If the line from *Chips* to *Cola* is thick, it means a very high percentage of people who bought Chips also threw Cola into their basket.

**4. Arrow Color (Lift)**
* The **color** of the line represents the rule's **Lift** (its statistical significance).
* **Light/Faint Red:** The lift is low (close to 1.0). The items are bought together, but it might just be a coincidence because both items are naturally very popular (like Bread and Milk).
* **Dark, Deep Red:** The lift is very high. This implies a strong, hidden correlation. The purchase of item A actively *drives* the purchase of item B.

---

### How to use the Interactive Sliders

The true power of this visualization comes from filtering out the noise to find the "golden rules":

* **`Min Support` (The Popularity Filter):** Increase this slider to filter out rare, obscure purchases. If you set it too high, you will only be left with the absolute most common staples (like Bread and Eggs).
* **`Min Confidence` (The Reliability Filter):** Increase this to remove weak rules. Setting this to `0.60` means the graph will only show an arrow if there is at least a 60% chance of the second item being bought.
* **`Min Lift` (The "Coincidence" Filter):** This is the most important slider! Setting `Min Lift` to $> 1.2$ or higher will instantly erase all the boring, coincidental rules. The graph will update to show only the highly correlated, actionable insights that store managers use to place items next to each other on the shelves!

### 2. FP-Growth (Frequent Pattern Growth)

Apriori is slow because of **Candidate Generation**. If you have 10,000 items in your store, Apriori has to mathematically generate and check millions of combinations ($A \rightarrow B$, $A \rightarrow C$, $A,B \rightarrow C$, etc.), scanning the database repeatedly.

FP-Growth completely eliminates candidate generation. Instead, it compresses the entire transaction database into a highly condensed tree structure called an **FP-Tree**. Once the tree is built, the algorithm recursively extracts frequent itemsets directly from the branches. It is exponentially faster and requires only **two scans** of the database.

**Mathematical Foundation (The 2-Scan Process):**
1. **First Scan (Frequency Counting):** Scan the database to find the support count of every single item. Discard any items below the `min_support` threshold. Sort the remaining items in descending order of frequency.
2. **Second Scan (Tree Building):** * Read the first transaction. Sort the items in that transaction based on the frequency list from Step 1.
   * Draw a path on the tree for those items.
   * Read the second transaction. If it shares a prefix with the first transaction, follow the existing path and increment the count by $1$. If it diverges, branch off and create a new node.
3. **Mining the Tree:** Start from the bottom of the tree and work upwards (creating conditional pattern bases) to find the final rules.

**Example Problem:**
* **Web Log Analysis:** A massive website like Amazon compressing millions of user click-streams into an FP-Tree to instantly figure out which pages are most frequently visited together, without running out of RAM.

In the visualization below, watch how multiple transactions that share the same popular items (like Milk and Bread) are compressed into a single, thick branch of the tree, saving massive amounts of memory!

* **`Min Count` (The Pruning Shear):** This slider controls the minimum support threshold. 
* If you set it to **1**, you will see every single item from the transactions, resulting in a wide, sprawling tree.
* If you increase it to **3** or **4**, watch how the tree instantly shrinks! The algorithm completely ignores rare, infrequent items (like *Beer* or *Diapers* in our sample) before it even builds the tree. This pruning makes extracting the final Association Rules lightning-fast.

In [6]:
from ipywidgets import IntSlider
class FPNode:
    def __init__(self, item, count, parent):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}

def build_fp_tree(transactions, min_support_count):
    item_counts = {}
    for basket in transactions:
        for item in basket:
            item_counts[item] = item_counts.get(item, 0) + 1
            
    frequent_items = {k: v for k, v in item_counts.items() if v >= min_support_count}
    
    root = FPNode('Null', 1, None)
    
    for basket in transactions:
        filtered_basket = [item for item in basket if item in frequent_items]
        filtered_basket.sort(key=lambda x: frequent_items[x], reverse=True)
        
        current_node = root
        for item in filtered_basket:
            if item in current_node.children:
                current_node.children[item].count += 1
            else:
                new_node = FPNode(item, 1, current_node)
                current_node.children[item] = new_node
            current_node = current_node.children[item]
            
    return root

def draw_tree(root):
    G = nx.DiGraph()
    labels = {}
    node_sizes = []
    
    def traverse(node, node_id="0"):
        label = f"{node.item}\n({node.count})"
        G.add_node(node_id, label=label, count=node.count)
        labels[node_id] = label
        node_sizes.append(node.count * 800)
        
        for i, (child_name, child_node) in enumerate(node.children.items()):
            child_id = f"{node_id}_{i}"
            G.add_edge(node_id, child_id)
            traverse(child_node, child_id)

    traverse(root)
    
    plt.figure(figsize=(12, 8))
    try:
        pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
    except:
        pos = nx.spring_layout(G, seed=42)
        
    nx.draw(G, pos, with_labels=True, labels=labels, node_color='lightgreen', 
            node_size=3000, font_size=10, font_weight='bold', edge_color='gray', arrows=True)
    
    plt.title("FP-Growth: Frequent Pattern Tree Compression", fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.show()

sample_transactions = [
    ['Milk', 'Bread', 'Beer'],
    ['Bread', 'Butter'],
    ['Milk', 'Bread', 'Butter'],
    ['Milk', 'Beer'],
    ['Bread', 'Milk', 'Butter', 'Beer'],
    ['Milk', 'Bread', 'Butter']
]

@interact(min_support_count=IntSlider(min=1, max=10, step=1, value=2, description='Min Count'))
def visualize_fp_tree(min_support_count):
    root = build_fp_tree(sample_transactions, min_support_count)
    draw_tree(root)

interactive(children=(IntSlider(value=2, description='Min Count', max=10, min=1), Output()), _dom_classes=('wi…

The FP-Tree (Frequent Pattern Tree) is the secret behind why the FP-Growth algorithm is so incredibly fast. Instead of scanning the database thousands of times to guess patterns, it scans the database exactly twice and builds a highly compressed map of every transaction.

In the FP-Growth algorithm, the items that are sold the most often across the entire store are intentionally forced to the very top of the tree. Before the algorithm even starts drawing the tree, it looks at all the receipts and counts the total number of times each item was sold.

* Milk: Sold 5 times
* Bread: Sold 4 times
* Butter: Sold 3 times
* Beer: Sold 2 times

Now, the algorithm takes every single customer's receipt and rearranges the items so they are always in order from most popular to least popular.

If a customer walks up to the register and their basket looks like this:
* `[Beer, Butter, Milk]`

The algorithm instantly rewrites their receipt to look like this:
* `[Milk, Butter, Beer]`

By forcing the most sold items (like `Milk` and `Bread`) to be at the front of every single receipt, the algorithm guarantees that when it draws the tree, those popular items will form a massive, shared "highway" at the very top.

If 10,000 people bought `Milk`, the tree only has to draw the `Milk` node once right at the top, and all 10,000 branches will sprout out from it. The rare items (like `Beer` or `Diapers`) are pushed to the very bottom, forming the tiny, individual "twigs" of the tree.

So when you look at the visualization and see `[Null -> Milk -> Bread -> Butter]`, it literally means: "These are the most sold items in the store, and they are bought together so often that they formed the thickest trunk of the tree."

Here is how to read the visual elements of the tree:

**1. The Root Node (`Null`)**
* Every FP-Tree starts with a base `Null` node. All transactions branch out from this single starting point.

**2. The Nodes (Green Bubbles)**
* Each bubble represents an **Item** and its **Frequency Count** at that specific point in the tree.
* **Top-Heavy Design:** Notice how the most popular items across the entire store (like *Milk* or *Bread*) are always pushed to the very top, right below `Null`. The algorithm sorts every transaction by global popularity before adding it to the tree. 

**3. The Paths (Branches & Compression)**
* Try tracing a line from `Null` down to the bottom of a branch. That path represents a specific sequence of items bought together.
* **The Magic of Compression:** If three different customers buy `[Milk, Bread, Beer]`, `[Milk, Bread, Butter]`, and `[Milk, Bread, Diapers]`, the FP-Tree does *not* create three separate branches from the top. Instead, it creates one thick trunk for `Null` $\rightarrow$ `Milk` $\rightarrow$ `Bread` and only splits at the very end. This shared prefixing is how FP-Growth compresses massive databases into a fraction of their original size!

---